### Standard Scaler

`StandardScaler` standardizes each numeric feature by removing its mean and scaling to unit variance (media ≈ 0, std ≈ 1), i.e.:



$$
z = \frac{x - \mu}{\sigma}
$$

**When to use**
- Features are on very different scales.
- Models are distance- or gradient-based (e.g., KNN, SVM, logistic/linear regression, neural networks, PCA).
- You want faster, more stable optimization and comparable feature magnitudes.

It is usually **not necessary** for tree-based models (Decision Trees, Random Forest, Gradient Boosting).

##### Quick usage (scikit-learn)

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
```

- Fit on training data only.
- Use the same fitted scaler to transform validation/test data.

### Min-Max Scaler

`MinMaxScaler` rescales each numeric feature to a fixed range (default $[0,1]$):

$$
x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

**When to use**
- You need features bounded to the same interval (commonly $[0,1]$).
- Models or pipelines benefit from fixed input ranges (e.g., neural networks, distance-based methods).
- You want to preserve the original shape/order of values while changing scale.

It can be sensitive to outliers, since extreme values set the min and max.

##### Quick usage (scikit-learn)

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_m = scaler.fit_transform(X_train)
X_test_m = scaler.transform(X_test)
```

- Fit on training data only.
- Use the same fitted scaler to transform validation/test data.

### Logarithmic Normalization

Log normalization compresses large values and reduces right-skew in long-tail distributions.

$$
\text{Natural log (base }e\text{):}\quad x' = \ln(x)
$$

$$
\text{Zero-safe version:}\quad x' = \ln(1+x)=\operatorname{log1p}(x)
$$

**When to use**
- Features are positive and highly right-skewed (e.g., income, counts, sales).
- Very large values dominate model behavior.
- You want to stabilize variance before applying scaling (often `log1p` then `StandardScaler`).

`\ln(x)` requires $x>0$, while `log1p(x)` also works when $x=0$.

##### Quick usage (NumPy)

```python
import numpy as np

X_train_log = np.log1p(X_train)
X_test_log = np.log1p(X_test)
```

- Apply the same transformation to train/validation/test.
- If negatives exist, shift or use an alternative transform first.

### Grid Search CV

`GridSearchCV` is a systematic way to find the best hyperparameters using cross-validation: it tries all parameter combinations and keeps the one with the best average validation score.

**When to use**
- You have a small/medium hyperparameter space and want a reliable baseline.
- You want a more objective selection than manual trial-and-error.
- You need stronger generalization estimates via cross-validation.

**Limitation**
- It can become computationally expensive when the parameter grid is large.

##### Quick usage (scikit-learn)

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
best_params = grid.best_params_
```

- `cv=3`: 3-fold cross-validation.
- `n_jobs=-1`: uses all CPU cores.
- `verbose=2`: prints detailed progress during search.

### PCA (Principal Component Analysis)

`PCA` is a dimensionality reduction method that transforms the original features into new uncorrelated components, ordered by how much variance they explain.

$$
Z = XW
$$

Where $W$ contains the principal directions (eigenvectors), and the first components keep most of the dataset information.

**When to use**
- You have many numeric features with strong correlation.
- You want to reduce dimensionality while preserving most variance.
- You need faster training or less noise before modeling.

**Limitation**
- Components are harder to interpret because they are linear combinations of original features.

##### Quick usage (scikit-learn)

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.90)  # keep ~90% explained variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
```

- Fit `PCA` on training data only.
- Apply the same fitted transform to validation/test sets.
- Standardize features before PCA in most cases.